# Fiche 4 — Le dictionnaire des données et le RGPD

| | |
|---|---|
| 🎯 **Objectif** | Remplir un dictionnaire des données complet et décider quoi faire d'une donnée personnelle |
| ⏱️ **Durée** | 30 min |
| 🧰 **Prérequis** | Fiches 1 à 3 |
| 🔗 **Dans le projet** | Partie 1, étape 1 (livrable 1 : le dictionnaire) |

> **L'idée clé.** Le dictionnaire est le **mode d'emploi** de la base : quelqu'un qui ne l'a pas construite doit pouvoir comprendre chaque colonne sans lire le code.

## 1. Les 7 colonnes, sur un autre exemple : une bibliothèque

| Colonne | Question à laquelle elle répond |
|---|---|
| Code | Comment s'appelle la colonne ? |
| Signification | Que contient-elle, en mots simples ? |
| Type | Texte, entier, décimal, date ? |
| Longueur | Combien de caractères au plus (pour un texte) ? |
| Nature | Élémentaire (lue telle quelle), calculée (produite par une règle) ou concaténée (collage de plusieurs données) ? |
| Règle de gestion | Quelle contrainte respecter (unique, non nulle, format…) ? |
| Règle de calcul | Comment l'obtient-on, si elle n'est pas élémentaire ? |

In [1]:
import pandas as pd

dico = pd.DataFrame([
    ["id_emprunt", "Identifiant de l'emprunt", "Entier", "NC", "Calculée", "Clé primaire, unique", "Numéro attribué 1, 2, 3…"],
    ["isbn", "Numéro international du livre", "Texte", "17", "Élémentaire", "Non nul ; format 978-…", ""],
    ["date_emprunt", "Date de sortie du livre", "Date", "10", "Élémentaire", "AAAA-MM-JJ ; non nulle", ""],
    ["date_retour_prevue", "Date limite de retour", "Date", "10", "Calculée", "Postérieure à date_emprunt", ""],
    ["code_exemplaire", "Identifiant d'un exemplaire précis", "Texte", "22", "Concaténée", "Unique", "isbn + '-' + numéro d'exemplaire"],
], columns=["CODE", "SIGNIFICATION", "TYPE", "LONGUEUR", "NATURE", "RÈGLE DE GESTION", "RÈGLE DE CALCUL"])
dico

,CODE,SIGNIFICATION,TYPE,LONGUEUR,NATURE,RÈGLE DE GESTION,RÈGLE DE CALCUL
0,id_emprunt,Identifiant de l'emprunt,Entier,NC,Calculée,"Clé primaire, unique","Numéro attribué 1, 2, 3…"
1,isbn,Numéro international du livre,Texte,17,Élémentaire,Non nul ; format 978-…,
2,date_emprunt,Date de sortie du livre,Date,10,Élémentaire,AAAA-MM-JJ ; non nulle,
3,date_retour_prevue,Date limite de retour,Date,10,Calculée,Postérieure à date_emprunt,
4,code_exemplaire,Identifiant d'un exemplaire précis,Texte,22,Concaténée,Unique,isbn + '-' + numéro d'exemplaire


## 2. ⚠️ Piège : la règle de calcul oubliée

In [2]:
oublis = dico[(dico["NATURE"] != "Élémentaire") & (dico["RÈGLE DE CALCUL"] == "")]
oublis[["CODE", "NATURE", "RÈGLE DE CALCUL"]]

,CODE,NATURE,RÈGLE DE CALCUL
3,date_retour_prevue,Calculée,


`date_retour_prevue` est **calculée**, mais rien ne dit comment (21 jours après l'emprunt ? 30 ?). L'an prochain,
personne ne saura la recalculer. Une donnée calculée ou concaténée a **toujours** sa règle de calcul.

## 3. Le RGPD : la colonne « Nom de l'acquéreur »

Le **RGPD** impose de ne traiter des données personnelles que si elles sont **nécessaires** au but poursuivi. Le
fichier des valeurs foncières contient le nom de l'acheteur (plus de 11 000 noms différents) ; aucun des besoins
d'analyse ne porte sur les acheteurs.

| Option | Ce qu'on fait | Quand c'est justifié |
|---|---|---|
| **Anonymiser** | supprimer la colonne | la donnée ne sert à rien : c'est le cas ici |
| **Pseudonymiser** | remplacer le nom par un identifiant (`acheteur_0042`) | on doit suivre un même acheteur dans le temps sans savoir qui il est |

Une donnée pseudonymisée reste une donnée personnelle au sens du RGPD (on peut la ré-identifier avec la table de
correspondance). Anonymiser est plus simple et plus sûr.

### ✍️ À toi de jouer

Ajoute au dictionnaire de la bibliothèque une colonne `jours_de_retard`, calculée à partir des dates. Remplis les 7 colonnes. Puis réponds : faut-il vraiment la **stocker** dans la base ?

**Ta réponse** (quelques phrases) : …

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```text
jours_de_retard | Nombre de jours entre la date limite et le retour réel | Entier | NC | Calculée |
Positif ou nul ; nul si rendu à temps | date_retour_reelle - date_retour_prevue (0 si négatif)
```

C'est une donnée **calculée** : la règle est obligatoire. La stocker, c'est risquer qu'elle ne soit plus à jour si une date change (une forme de redondance, fiche 2) ; on la recalcule plutôt dans une requête.
</details>

## ✅ À retenir

1. Sept colonnes : code, signification, type, longueur, nature, règle de gestion, règle de calcul.
2. Une donnée calculée ou concaténée a **toujours** sa règle de calcul.
3. RGPD : ne garder une donnée personnelle que si elle est nécessaire ; ici, on supprime le nom de l'acquéreur.
4. Pseudonymiser n'est pas anonymiser : la donnée reste personnelle.